user-research-synthesis

In [ ]:
import sys, json, types
lrn_llm = types.ModuleType("lrn_llm")
try:
    from pyodide.http import pyfetch as _pyfetch
    _IN_PYODIDE = True
except ImportError:
    import urllib.request as _urlreq
    _IN_PYODIDE = False
lrn_llm.API_BASE = "/api/llm"  # same-origin proxy; server injects the gateway key
lrn_llm.DEFAULT_MODEL = "azure/gpt-5.4-mini"
lrn_llm.API_KEY = ""  # optional; set in Step 0a

async def _lrn_call(messages, *, system=None, max_tokens=400, model=None):
    if system is not None:
        messages = [{"role": "system", "content": system}] + list(messages)
    payload = {"model": model or lrn_llm.DEFAULT_MODEL, "messages": messages,
               "max_completion_tokens": max_tokens}
    headers = {"content-type": "application/json"}
    _key = lrn_llm.API_KEY
    if _key:
        headers["Authorization"] = "Bearer " + _key
    url = lrn_llm.API_BASE.rstrip("/") + "/chat/completions"
    body = json.dumps(payload)
    if _IN_PYODIDE:
        r = await _pyfetch(url, method="POST", headers=headers, body=body)
        data = await r.json()
    else:
        req = _urlreq.Request(url, method="POST", headers=headers, data=body.encode("utf-8"))
        with _urlreq.urlopen(req, timeout=60) as r:
            data = json.loads(r.read())
    if "error" in data:
        raise RuntimeError("LLM error: " + str(data["error"]))
    return data

def _lrn_text(r):
    ch = (r or {}).get("choices") or []
    return (ch[0].get("message", {}) or {}).get("content", "") if ch else ""

async def _lrn_ping():
    r = await _lrn_call([{"role": "user", "content": "Reply with exactly: OK"}], max_tokens=5)
    return {"ok": _lrn_text(r).strip().upper().startswith("OK"), "model": r.get("model")}

lrn_llm.call = _lrn_call
lrn_llm.text = _lrn_text
lrn_llm.ping = _lrn_ping
print("✅ notebook ready · endpoint:", lrn_llm.API_BASE)

## Step 0a — Endpoint & Key

Set your API key (optional on LHIND network) and verify the endpoint.

In [ ]:
# Optional: Set your API key if not on LHIND network
lrn_llm.API_KEY = ""

print(f"Endpoint: {lrn_llm.API_BASE}")
print(f"Model:    {lrn_llm.DEFAULT_MODEL}")
print(f"Key set:  {bool(lrn_llm.API_KEY)}")

## Step 1 — Reachability

Test the LLM endpoint to ensure connectivity.

In [ ]:
r = await lrn_llm.ping()
print(f"✅ LLM reachable: {r['ok']} (model: {r.get('model', 'unknown')})")

## Step 2 — Schema for Interview Synthesis

We'll extract structured insights from user research interviews. The schema defines what we want: a persona type, the main theme discussed, the emotional tone, and direct evidence from the text.

In [ ]:
# Define the interview synthesis schema
synthesis_schema = {
    "type": "object",
    "properties": {
        "persona": {
            "type": "string",
            "enum": ["early_adopter", "skeptic", "power_user", "casual_user"]
        },
        "theme": {
            "type": "string",
            "description": "Main topic or pain point discussed"
        },
        "sentiment": {
            "type": "string",
            "enum": ["positive", "neutral", "negative"]
        },
        "evidence": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Direct quotes supporting the analysis",
            "minItems": 1,
            "maxItems": 3
        }
    },
    "required": ["persona", "theme", "sentiment", "evidence"]
}

print("Interview Synthesis Schema:")
print(json.dumps(synthesis_schema, indent=2))

## Step 3 — Schema Validation

Before we call the LLM, let's build a validator to check that responses match our schema. This is the foundation of reliable structured outputs.

In [ ]:
def validate_schema(data, schema):
    """Validate that data matches the schema. Return list of error messages."""
    errors = []
    
    def _validate(value, rule, path):
        rule_type = rule.get("type")
        
        if rule_type == "object":
            if not isinstance(value, dict):
                errors.append(f"{path}: expected object, got {type(value).__name__}")
                return
            for required_key in rule.get("required", []):
                if required_key not in value:
                    errors.append(f"{path}.{required_key}: required field missing")
            for key, val in value.items():
                if key in rule.get("properties", {}):
                    _validate(val, rule["properties"][key], f"{path}.{key}")
        
        elif rule_type == "array":
            if not isinstance(value, list):
                errors.append(f"{path}: expected array, got {type(value).__name__}")
                return
            min_items = rule.get("minItems", 0)
            max_items = rule.get("maxItems", float("inf"))
            if len(value) < min_items:
                errors.append(f"{path}: needs at least {min_items} items, got {len(value)}")
            if len(value) > max_items:
                errors.append(f"{path}: max {max_items} items, got {len(value)}")
            items_rule = rule.get("items", {})
            for i, item in enumerate(value):
                _validate(item, items_rule, f"{path}[{i}]")
        
        elif rule_type == "string":
            if not isinstance(value, str):
                errors.append(f"{path}: expected string, got {type(value).__name__}")
                return
            if "enum" in rule and value not in rule["enum"]:
                errors.append(f"{path}: '{value}' not in {rule['enum']}")
    
    _validate(data, schema, "root")
    return errors

# Test the validator
test_valid = {
    "persona": "power_user",
    "theme": "API reliability",
    "sentiment": "positive",
    "evidence": ["Works great", "Fast response times"]
}

test_invalid = {
    "persona": "unknown_type",
    "theme": "Something",
    "sentiment": "positive"
}

print("Valid synthesis:", validate_schema(test_valid, synthesis_schema))
print("Invalid synthesis:", validate_schema(test_invalid, synthesis_schema))

## Step 4 — Extract from Interview Snippet

Now we use the LLM to synthesize a user research interview into our structured format. The prompt tells the model: "Return valid JSON matching this schema."

In [ ]:
async def synthesize_interview(interview_text):
    """Call LLM to extract structured insights from interview snippet."""
    
    system_prompt = f"""You are a user research analyst. Extract key insights from the interview snippet as valid JSON matching this schema:
{json.dumps(synthesis_schema, indent=2)}

IMPORTANT:
- 'persona' must be one of: early_adopter, skeptic, power_user, casual_user
- 'sentiment' must be one of: positive, neutral, negative
- 'evidence' must be a list of 1-3 direct quotes from the text
- Return ONLY valid JSON, no markdown or explanation"""
    
    response = await lrn_llm.call(
        [{"role": "user", "content": interview_text}],
        system=system_prompt,
        max_tokens=500
    )
    
    raw_text = lrn_llm.text(response)
    
    try:
        data = json.loads(raw_text)
        errors = validate_schema(data, synthesis_schema)
        if errors:
            return {"success": False, "data": data, "errors": errors, "raw": raw_text}
        return {"success": True, "data": data, "errors": [], "raw": raw_text}
    except json.JSONDecodeError as e:
        return {"success": False, "data": None, "errors": [f"JSON parse error: {e}"], "raw": raw_text}

# Test with a sample interview
interview_1 = """Interviewer: How do you use the product?
User: I've been testing beta features for years. The new API is exactly what I needed. 
Previously we had to write custom integration code, now it's straightforward. 
My only wish is the docs were more detailed.
"""

print("Calling LLM to synthesize interview...")
result = await synthesize_interview(interview_1)

if result["success"]:
    print("✅ Synthesis successful")
    print(json.dumps(result["data"], indent=2))
else:
    print("❌ Synthesis failed")
    print("Errors:", result["errors"])
    print("Raw response:", result["raw"])

## Step 5 — Batch Synthesis with Retry Logic

In production, we often get malformed outputs. Let's build a retry mechanism: if the LLM fails validation, send the errors back and ask it to fix them.

In [ ]:
async def synthesize_interview_with_retry(interview_text, max_retries=2):
    """Synthesize with automatic retry on validation failure."""
    
    base_system = f"""You are a user research analyst. Extract key insights as valid JSON:
{json.dumps(synthesis_schema, indent=2)}
Return ONLY valid JSON."""
    
    for attempt in range(max_retries):
        user_msg = interview_text
        system_msg = base_system
        
        if attempt > 0:
            user_msg = f"""Previous response had errors. Please fix and return valid JSON.

Errors: {result['errors']}

Original interview:
{interview_text}"""
        
        response = await lrn_llm.call(
            [{"role": "user", "content": user_msg}],
            system=system_msg,
            max_tokens=500
        )
        
        raw_text = lrn_llm.text(response)
        
        try:
            data = json.loads(raw_text)
            errors = validate_schema(data, synthesis_schema)
            if not errors:
                return {"success": True, "data": data, "attempt": attempt + 1}
            result = {"data": data, "errors": errors, "raw": raw_text}
        except json.JSONDecodeError as e:
            result = {"data": None, "errors": [f"JSON parse error: {e}"], "raw": raw_text}
    
    return {"success": False, "data": None, "errors": result["errors"], "attempt": max_retries}

# Test batch synthesis
interviews = [
    """User: I'm skeptical about cloud products. Too many security concerns. 
We looked at your offering but the compliance docs were vague. 
We'll stick with our on-premise setup.""",
    """User: I just started using this. The UI is confusing at first, 
but once you know the shortcuts it's fast. My team loves the collaboration features."""
]

print("Batch processing interviews...\n")
for i, interview in enumerate(interviews, 1):
    print(f"Interview {i}:")
    result = await synthesize_interview_with_retry(interview)
    if result["success"]:
        print(f"  ✅ Success (attempt {result['attempt']})")
        print(f"  Persona: {result['data']['persona']}")
        print(f"  Theme: {result['data']['theme']}")
        print(f"  Sentiment: {result['data']['sentiment']}")
    else:
        print(f"  ❌ Failed after {result['attempt']} attempts")
        print(f"  Errors: {result['errors']}")
    print()

## Step 6 — Why Schema Validation Matters

Without structured outputs, the LLM returns free text. With structured outputs + validation, we get typed objects ready to score and aggregate. This is the difference between data science and data collection.

In [ ]:
# Compare: unstructured vs. structured

unstructured_response = """The user is clearly a technical person who values reliability. 
They were happy with the API but suggested better documentation. 
Overall positive impression."""

structured_response = {
    "persona": "power_user",
    "theme": "API documentation",
    "sentiment": "positive",
    "evidence": ["happy with the API", "suggested better documentation"]
}

print("Unstructured response (string):")
print(f"  {unstructured_response}")
print(f"  Type: {type(unstructured_response)}")
print(f"  Can aggregate personas? No")
print(f"  Can compute sentiment distribution? No")
print(f"  Can generate reports? Difficult\n")

print("Structured response (object):")
print(f"  {json.dumps(structured_response, indent=2)}")
print(f"  Type: {type(structured_response)}")
print(f"  Can aggregate personas? Yes (count each type)")
print(f"  Can compute sentiment distribution? Yes (group by sentiment)")
print(f"  Can generate reports? Easy (filter by theme, aggregate by persona)")

## Try it yourself

Edit the interview snippet below and run the synthesis. Experiment with different interview styles to see how the model classifies persona, theme, and sentiment.

In [ ]:
# TODO: Edit this interview snippet and run the synthesis
my_interview = """Interviewer: Tell us about your experience with the product.
User: It's been a game-changer for our workflow. The automation saved us hours every week.
We did hit some issues with the integration setup, but support got us sorted quickly.
Pretty happy overall."""

print("Running synthesis on your interview...\n")
result = await synthesize_interview_with_retry(my_interview, max_retries=2)

if result["success"]:
    print("✅ Synthesis complete\n")
    print(json.dumps(result["data"], indent=2))
    print(f"\nValidation passed on attempt {result['attempt']}")
else:
    print("❌ Synthesis failed")
    print(f"Errors: {result['errors']}")
    print(f"\nTip: Make sure your interview mentions a feeling (positive/negative/neutral)")
    print("and a specific persona signal (beginner, expert, etc.)")